# 异象检验完整教程：从异象发现到 Alpha 检验

## 📚 教学目标
1. 理解**异象 (Anomaly)** 的定义及其在因子投资中的核心地位
2. 掌握异象检验的**完整逻辑**：排序 → 构建多空组合 → 回归 → 检验 Alpha
3. 手算 **OLS 回归**获取 Alpha，理解每一步计算
4. 区分**真异象**（显著 Alpha）和**伪异象**（Alpha 不显著）
5. 理解为什么异象检验必须**控制已知因子**

**参考书**：《因子投资：方法与实践》（石川）第 2.4 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 什么是异象？为什么要做异象检验？

### 🎯 一个直觉问题

假设你发现了一个新的选股信号（比如"过去 12 个月动量"），按这个信号做多空组合，每月赚 0.8%。

但有人质疑："你的收益只是因为你买了小市值和高 B/M 的股票，跟动量没关系。"

**异象检验就是要回答这个问题**：控制了已知因子后，你的新信号还能赚钱吗？

### 📖 书中的定义

> 异象 (Anomaly) 是指某个变量或策略产生的收益无法被当前因子模型所解释。如果一个新变量在控制已知因子后仍然产生显著的 Alpha，则该变量构成一个异象。

### 📐 异象检验的核心逻辑

```
步骤 1: 发现新变量（如动量、换手率、应计利润...）
   ↓
步骤 2: 按新变量排序 → 构建多空组合 → 得到 Spread 收益时间序列
   ↓
步骤 3: 将 Spread 收益对已知因子模型做回归
        R_spread,t = α + β₁·MKT_t + β₂·SMB_t + β₃·HML_t + ε_t
   ↓
步骤 4: 检验 α 是否显著不为零
        ├── α 显著 → ✅ 异象成立（新变量包含已知因子无法解释的信息）
        └── α 不显著 → ❌ 不是异象（收益可以被已知因子解释）
```

### 💡 关键直觉

| 概念 | 含义 |
|------|------|
| Alpha (α) | 控制因子后的**超额收益**，即因子模型无法解释的部分 |
| β × Factor | 已知因子**能解释**的收益部分 |
| α 显著 | 新变量有独立的选股能力，构成异象 |
| α 不显著 | 新变量的选股能力完全来自已知因子的暴露 |

In [ ]:
import sys, os
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("❌ matplotlib 未安装! 请运行: !pip install matplotlib seaborn statsmodels scipy")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 异象的定义与直觉：10 只股票的微型示例

### 🎯 场景

假设市场上只有 **10 只股票**，我们已知 Fama-French 三因子模型（MKT, SMB, HML）。

现在有人提出一个新变量："**过去 12 个月动量**"（过去涨幅）可能是一个有效的选股信号。

我们要检验：控制 FF3 因子后，动量是否还能产生显著的 Alpha？

### 📐 数据生成逻辑

每只股票的收益率 $r_i$ 由以下部分组成：

$$r_i = \beta_{\text{MKT},i} \cdot \text{MKT} + \beta_{\text{SMB},i} \cdot \text{SMB} + \beta_{\text{HML},i} \cdot \text{HML} + \gamma \cdot \text{Momentum}_i + \varepsilon_i$$

其中：
- $\text{MKT}, \text{SMB}, \text{HML}$ = 已知的三个因子（对所有股票相同）
- $\text{Momentum}_i$ = 股票 $i$ 的动量信号（每只股票不同）
- $\gamma$ = 动量对收益的真实影响系数
  - $\gamma > 0$ → 动量是**真异象**（控制 FF3 后仍有 Alpha）
  - $\gamma = 0$ → 动量是**伪异象**（没有独立的选股能力）

In [ ]:
# ========== 构造 10 只股票的微型数据 ==========
np.random.seed(42)

# 10 只股票的动量信号（过去12个月收益率）
stocks = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']
momentum = np.array([-15, -10, -5, -2, 0, 3, 5, 10, 15, 25])  # 过去收益率 (%)

# 本期 FF3 因子收益（市场层面，所有股票面临相同的因子）
MKT = 1.2   # 市场超额收益 1.2%
SMB_val = 0.5   # 小市值溢价 0.5%
HML_val = -0.3  # 价值因子 -0.3%

# 每只股票的因子暴露（beta）
beta_mkt = np.array([0.8, 1.0, 0.9, 1.2, 1.1, 0.7, 1.0, 1.3, 0.9, 1.1])
beta_smb = np.array([0.5, 0.3, -0.2, 0.1, 0.4, -0.3, 0.2, 0.6, -0.1, 0.3])
beta_hml = np.array([0.2, -0.1, 0.4, 0.3, -0.2, 0.5, 0.1, -0.3, 0.2, 0.4])

# 动量的真实效应系数
gamma = 0.08  # 每 1% 的过去动量 → 本期多赚 0.08%

# 噪声
noise = np.random.normal(0, 1.5, 10)

# 收益率 = 因子暴露 × 因子收益 + 动量效应 + 噪声
returns = (beta_mkt * MKT + beta_smb * SMB_val + beta_hml * HML_val
           + gamma * momentum + noise)

# 构建 DataFrame
df_micro = pd.DataFrame({
    'Stock': stocks,
    'Momentum (%)': momentum,
    'Return (%)': np.round(returns, 2),
    'beta_MKT': beta_mkt,
    'beta_SMB': beta_smb,
    'beta_HML': beta_hml
})

print("📋 10 只股票数据集：")
print(df_micro.to_string(index=False))
print(f"\n💡 观察：动量越高的股票（S09, S10），本期收益倾向越高")
print(f"   动量效应系数 γ = {gamma}（每 1% 过去动量 → 本期多赚 {gamma}%）")
print(f"   本期因子: MKT={MKT}%, SMB={SMB_val}%, HML={HML_val}%")

In [ ]:
# ========== 步骤 1: 按动量排序分组 ==========
print("📊 步骤 1: 按动量信号排序分组")
print("─" * 55)

# 分为 2 组：Low Momentum (Q1) vs High Momentum (Q2)
df_sorted = df_micro.sort_values('Momentum (%)').reset_index(drop=True)

# 前 5 只 = Low Momentum, 后 5 只 = High Momentum
df_sorted['Group'] = ['Low'] * 5 + ['High'] * 5

print("\n  Low Momentum 组（做空）:")
for _, row in df_sorted[df_sorted['Group'] == 'Low'].iterrows():
    print(f"    {row['Stock']}: 动量 = {row['Momentum (%)']:>5}%,  收益 = {row['Return (%)']:>6.2f}%")

print("\n  High Momentum 组（做多）:")
for _, row in df_sorted[df_sorted['Group'] == 'High'].iterrows():
    print(f"    {row['Stock']}: 动量 = {row['Momentum (%)']:>5}%,  收益 = {row['Return (%)']:>6.2f}%")

# ========== 步骤 2: 计算多空收益 ==========
ret_high = df_sorted[df_sorted['Group'] == 'High']['Return (%)'].mean()
ret_low = df_sorted[df_sorted['Group'] == 'Low']['Return (%)'].mean()
spread = ret_high - ret_low

print(f"\n📊 步骤 2: 构建多空组合")
print(f"  R_High = {ret_high:.2f}%")
print(f"  R_Low  = {ret_low:.2f}%")
print(f"  Spread = R_High - R_Low = {ret_high:.2f} - {ret_low:.2f} = {spread:.2f}%")
print(f"\n💡 单期 Spread = {spread:.2f}%")
print(f"   但这还不是 Alpha！我们需要检验：控制 FF3 因子后，这个收益是否仍然显著")

---

## 3. Alpha 检验 — 核心方法

### 📐 时间序列回归

异象检验的核心是一个**时间序列回归**。将异象多空组合的收益序列对已知因子做回归：

$$R_{\text{anomaly},t} = \alpha + \beta_{\text{MKT}} \cdot \text{MKT}_t + \beta_{\text{SMB}} \cdot \text{SMB}_t + \beta_{\text{HML}} \cdot \text{HML}_t + \varepsilon_t$$

其中：
- $R_{\text{anomaly},t}$ = 异象多空组合在 $t$ 月的收益率
- $\text{MKT}_t, \text{SMB}_t, \text{HML}_t$ = FF3 因子在 $t$ 月的收益
- $\alpha$ = **截距项 (Alpha)** = 因子模型无法解释的超额收益
- $\beta$ = 异象组合对各因子的暴露
- $\varepsilon_t$ = 残差（噪声）

### 💡 为什么 Alpha 是关键？

```
Spread 收益 = α + β₁·MKT + β₂·SMB + β₃·HML + ε
              ↑       ↑           ↑           ↑
         无法解释   市场因子     市值因子    价值因子
         的部分     能解释的     能解释的    能解释的

如果 α = 0: Spread 完全来自对 FF3 的暴露 → 不是异象
如果 α ≠ 0: Spread 中有 FF3 无法解释的部分 → 异象成立！
```

### 📐 Alpha 的 T 检验

$$t_\alpha = \frac{\hat{\alpha}}{\text{SE}(\hat{\alpha})}$$

其中 $\text{SE}(\hat{\alpha})$ 是 Alpha 估计值的标准误差。

**判断标准**：$|t_\alpha| > 2$（经验法则），即 $\alpha$ 显著不为零。

In [ ]:
# ========== 模拟 60 个月的因子收益和异象组合收益 ==========
np.random.seed(42)
T = 60  # 60 个月

# FF3 因子月度收益（市场层面）
# MKT ~ N(0.5, 4), SMB ~ N(0.2, 2), HML ~ N(0.3, 2)
MKT_series = np.random.normal(0.5, 4, T)
SMB_series = np.random.normal(0.2, 2, T)
HML_series = np.random.normal(0.3, 2, T)

# 异象组合对 FF3 的真实暴露
true_beta_mkt = 0.3    # 对市场有一定正暴露
true_beta_smb = 0.5    # 对 SMB 有正暴露（偏小市值）
true_beta_hml = -0.2   # 对 HML 有负暴露（偏成长）

# 真实 Alpha（异象组合的真实超额收益）
true_alpha = 0.8  # 每月 0.8% 的 Alpha

# 异象组合收益 = α + β·Factors + ε
noise_series = np.random.normal(0, 2, T)
R_anomaly = (true_alpha
             + true_beta_mkt * MKT_series
             + true_beta_smb * SMB_series
             + true_beta_hml * HML_series
             + noise_series)

print(f"📊 模拟参数：")
print(f"  时间跨度: {T} 个月")
print(f"  FF3 因子:")
print(f"    MKT ~ N(0.5, 4²), 月均 = {MKT_series.mean():.3f}%")
print(f"    SMB ~ N(0.2, 2²), 月均 = {SMB_series.mean():.3f}%")
print(f"    HML ~ N(0.3, 2²), 月均 = {HML_series.mean():.3f}%")
print(f"  异象组合真实参数:")
print(f"    α = {true_alpha}% (每月超额收益)")
print(f"    β_MKT = {true_beta_mkt}, β_SMB = {true_beta_smb}, β_HML = {true_beta_hml}")
print(f"  异象组合月均收益 = {R_anomaly.mean():.3f}%")
print(f"\n💡 异象组合收益 = {true_alpha} + 0.3·MKT + 0.5·SMB − 0.2·HML + ε")
print(f"   我们的目标：从数据中把 α = {true_alpha}% 估计出来")

### 📐 手算 OLS 回归

OLS（普通最小二乘法）回归的矩阵形式：

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

其中：
- $\mathbf{y}$ = 异象组合收益向量 ($T \times 1$)
- $\mathbf{X}$ = 自变量矩阵 ($T \times 4$)，含截距项和三个因子
- $\hat{\boldsymbol{\beta}} = [\hat{\alpha}, \hat{\beta}_{\text{MKT}}, \hat{\beta}_{\text{SMB}}, \hat{\beta}_{\text{HML}}]^\top$

Alpha 的标准误差：

$$\text{SE}(\hat{\alpha}) = \hat{\sigma} \cdot \sqrt{[(\mathbf{X}^\top \mathbf{X})^{-1}]_{1,1}}$$

其中 $\hat{\sigma}^2 = \frac{\sum \hat{\varepsilon}_t^2}{T - k}$，$k$ 是参数个数（含截距）。

In [ ]:
# ========== 手算 OLS 回归 ==========
print("📊 步骤 1: 构造矩阵")
print("─" * 55)

# y 向量
y = R_anomaly.copy()

# X 矩阵: [1, MKT, SMB, HML]
ones = np.ones(T)
X = np.column_stack([ones, MKT_series, SMB_series, HML_series])

print(f"  y 向量: {T} × 1")
print(f"  X 矩阵: {T} × {X.shape[1]}")
print(f"  X 的前 3 行:")
print(f"    {'Const':>8s}  {'MKT':>8s}  {'SMB':>8s}  {'HML':>8s}")
for i in range(3):
    print(f"    {X[i,0]:>8.2f}  {X[i,1]:>8.3f}  {X[i,2]:>8.3f}  {X[i,3]:>8.3f}")
print(f"    ...")

# ========== 步骤 2: 计算 (X'X)^{-1} X'y ==========
print(f"\n📊 步骤 2: 计算 OLS 系数 β̂ = (X'X)⁻¹ X'y")
print("─" * 55)

XtX = X.T @ X
print(f"  X'X ({XtX.shape[0]}×{XtX.shape[1]} 矩阵):")
for row in XtX:
    print(f"    [{', '.join([f'{v:>10.2f}' for v in row])}]")

XtX_inv = np.linalg.inv(XtX)
Xty = X.T @ y
beta_hat = XtX_inv @ Xty

print(f"\n  📐 β̂ = (X'X)⁻¹ X'y:")
param_names = ['α (Intercept)', 'β_MKT', 'β_SMB', 'β_HML']
true_vals = [true_alpha, true_beta_mkt, true_beta_smb, true_beta_hml]
for name, est, true in zip(param_names, beta_hat, true_vals):
    print(f"    {name:>16s} = {est:>8.4f}  (真实值: {true})")

# ========== 步骤 3: 计算残差和 Alpha 标准误 ==========
print(f"\n📊 步骤 3: 计算残差和标准误")
print("─" * 55)

# 残差
y_hat = X @ beta_hat
residuals = y - y_hat

# 残差标准差
k = X.shape[1]  # 参数个数 = 4
sigma_hat_sq = np.sum(residuals**2) / (T - k)
sigma_hat = np.sqrt(sigma_hat_sq)

print(f"  残差平方和 RSS = Σε̂² = {np.sum(residuals**2):.4f}")
print(f"  自由度 = T - k = {T} - {k} = {T - k}")
print(f"  σ̂² = RSS / (T-k) = {sigma_hat_sq:.4f}")
print(f"  σ̂ = {sigma_hat:.4f}")

# Alpha 的标准误
se_alpha = sigma_hat * np.sqrt(XtX_inv[0, 0])
print(f"\n  (X'X)⁻¹ 的 [1,1] 元素 = {XtX_inv[0,0]:.6f}")
print(f"  SE(α̂) = σ̂ × √{XtX_inv[0,0]:.6f}")
print(f"         = {sigma_hat:.4f} × {np.sqrt(XtX_inv[0,0]):.6f}")
print(f"         = {se_alpha:.4f}")

# ========== 步骤 4: T 检验 ==========
print(f"\n📊 步骤 4: Alpha 的 T 检验")
print("─" * 55)

t_alpha = beta_hat[0] / se_alpha
p_alpha = 2 * (1 - stats.t.cdf(abs(t_alpha), df=T-k))

print(f"  α̂ = {beta_hat[0]:.4f}%")
print(f"  SE(α̂) = {se_alpha:.4f}")
print(f"  t_α = α̂ / SE(α̂) = {beta_hat[0]:.4f} / {se_alpha:.4f} = {t_alpha:.4f}")
print(f"  P 值 = {p_alpha:.6f}")
print(f"")
if abs(t_alpha) > 2:
    print(f"  ✅ |t_α| = {abs(t_alpha):.2f} > 2")
    print(f"  ✅ Alpha 显著不为零 → 异象成立！")
    print(f"  ✅ 控制 FF3 后，该异象组合每月仍有 {beta_hat[0]:.2f}% 的超额收益")
else:
    print(f"  ❌ |t_α| = {abs(t_alpha):.2f} ≤ 2")
    print(f"  ❌ Alpha 不显著 → 不能确认为异象")

In [ ]:
# ========== 用 statsmodels 验证 ==========
X_sm = sm.add_constant(np.column_stack([MKT_series, SMB_series, HML_series]))
model = sm.OLS(R_anomaly, X_sm).fit()

print("🔬 statsmodels OLS 验证:")
print("─" * 55)
print(f"  {'参数':>14s}  {'手算':>10s}  {'statsmodels':>12s}")
print(f"  {'─'*14}  {'─'*10}  {'─'*12}")

sm_names = ['α (Intercept)', 'β_MKT', 'β_SMB', 'β_HML']
for name, manual, sm_val in zip(sm_names, beta_hat, model.params):
    print(f"  {name:>14s}  {manual:>10.4f}  {sm_val:>12.4f}")

print(f"\n  Alpha 检验:")
print(f"  {'':>14s}  {'手算':>10s}  {'statsmodels':>12s}")
print(f"  {'SE(α)':>14s}  {se_alpha:>10.4f}  {model.bse[0]:>12.4f}")
print(f"  {'t(α)':>14s}  {t_alpha:>10.4f}  {model.tvalues[0]:>12.4f}")
print(f"  {'P-value':>14s}  {p_alpha:>10.6f}  {model.pvalues[0]:>12.6f}")
print(f"\n  ✅ 验证通过！手算结果与 statsmodels 完全一致")
print(f"\n  R² = {model.rsquared:.4f}")
print(f"  💡 R² = {model.rsquared:.2%} 的收益变异能被 FF3 解释，剩下的包含 Alpha")

In [ ]:
# ========== 可视化: 异象组合收益与 Alpha ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

months = np.arange(1, T + 1)

# --- 图1: 异象组合收益时间序列 ---
ax1 = axes[0]
ax1.bar(months, R_anomaly,
        color=['#2ecc71' if r > 0 else '#e74c3c' for r in R_anomaly],
        alpha=0.7, edgecolor='none')
ax1.axhline(y=R_anomaly.mean(), color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {R_anomaly.mean():.2f}%')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('Anomaly Spread Return (%)', fontsize=11)
ax1.set_title('Anomaly Portfolio Returns', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 图2: 回归拟合 vs 实际 ---
ax2 = axes[1]
ax2.scatter(y_hat, R_anomaly, c='steelblue', alpha=0.7, edgecolors='black', s=50)
fit_range = np.linspace(min(y_hat), max(y_hat), 100)
ax2.plot(fit_range, fit_range, 'r--', linewidth=2, label='Perfect Fit (45-degree)')
ax2.set_xlabel('Predicted Return (%)', fontsize=11)
ax2.set_ylabel('Actual Return (%)', fontsize=11)
ax2.set_title(f'OLS Fit (R\u00b2 = {model.rsquared:.3f})', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

# --- 图3: 回归系数与 t 值 ---
ax3 = axes[2]
coef_names = ['\u03b1', '\u03b2_MKT', '\u03b2_SMB', '\u03b2_HML']
t_values = model.tvalues
colors_t = ['#e74c3c' if abs(t) > 2 else '#95a5a6' for t in t_values]
bars = ax3.barh(coef_names, t_values, color=colors_t, edgecolor='black', alpha=0.85)
ax3.axvline(x=2, color='red', linestyle='--', alpha=0.5, label='t = +2')
ax3.axvline(x=-2, color='red', linestyle='--', alpha=0.5, label='t = -2')
ax3.axvline(x=0, color='black', linewidth=0.8)

# 标注数值
for bar, tv in zip(bars, t_values):
    offset = 0.15 * np.sign(tv)
    ax3.text(tv + offset, bar.get_y() + bar.get_height()/2,
             f'{tv:.2f}', ha='left' if tv >= 0 else 'right', va='center', fontweight='bold')

ax3.set_xlabel('T-statistic', fontsize=11)
ax3.set_title('Regression Coefficients: T-values', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：异象多空组合的月度收益，均值 {R_anomaly.mean():.2f}% 偏离零线")
print(f"  图2：OLS 拟合效果，R\u00b2 = {model.rsquared:.3f}")
print(f"  图3：红色 = 显著 (|t|>2)；灰色 = 不显著。α 的 t 值 = {t_alpha:.2f}")

---

## 4. 完整流程：200 只股票 × 60 个月

### 🎯 完整异象检验

上面我们直接给定了异象组合的收益序列。现在我们**从头开始**：

1. 模拟 200 只股票、60 个月的截面数据
2. 每月按异象变量排序 → 构建多空组合
3. 收集 60 个月的 Spread
4. 将 Spread 对 FF3 做回归 → 检验 Alpha

我们同时检验两个变量：
- **真异象**：动量变量，$\gamma = 0.5$（有独立于 FF3 的选股能力）
- **伪异象**：随机变量，$\gamma = 0$（没有任何选股能力）

### 📐 DGP (数据生成过程)

$$r_{i,t} = \beta_{\text{MKT},i} \cdot \text{MKT}_t + \beta_{\text{SMB},i} \cdot \text{SMB}_t + \beta_{\text{HML},i} \cdot \text{HML}_t + \gamma \cdot \text{anomaly}_{i,t} + \varepsilon_{i,t}$$

其中：
- $\text{MKT}_t \sim N(0.5, 4^2)$, $\text{SMB}_t \sim N(0.2, 2^2)$, $\text{HML}_t \sim N(0.3, 2^2)$
- $\beta_{\text{MKT},i} \sim U(0.5, 1.5)$, $\beta_{\text{SMB},i} \sim U(-0.5, 0.5)$, $\beta_{\text{HML},i} \sim U(-0.5, 0.5)$
- $\text{anomaly}_{i,t} \sim N(0, 1)$（标准化的异象变量值）
- $\gamma = 0.5$（真异象）或 $\gamma = 0$（伪异象）
- $\varepsilon_{i,t} \sim N(0, 5^2)$（个股噪声）

In [ ]:
# ========== 完整 DGP: 200 只股票 × 60 个月 ==========
np.random.seed(42)
N_STOCKS = 200
N_MONTHS = 60
N_GROUPS = 5

# FF3 因子月度收益
MKT_f = np.random.normal(0.5, 4, N_MONTHS)
SMB_f = np.random.normal(0.2, 2, N_MONTHS)
HML_f = np.random.normal(0.3, 2, N_MONTHS)

# 每只股票的因子暴露（固定不变）
beta_mkt_i = np.random.uniform(0.5, 1.5, N_STOCKS)
beta_smb_i = np.random.uniform(-0.5, 0.5, N_STOCKS)
beta_hml_i = np.random.uniform(-0.5, 0.5, N_STOCKS)

# 真异象效应系数
gamma_true = 0.5   # 真异象
gamma_fake = 0.0   # 伪异象

# 存储每月的 Spread
true_anomaly_spreads = []
fake_anomaly_spreads = []

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  分组数量: {N_GROUPS} 组")
print(f"  FF3 因子: MKT~N(0.5,4²), SMB~N(0.2,2²), HML~N(0.3,2²)")
print(f"  真异象 γ = {gamma_true}, 伪异象 γ = {gamma_fake}")
print(f"")
print(f"🔄 开始逐月模拟...")

for t in range(N_MONTHS):
    # 本月异象变量（截面变异）
    anomaly_signal = np.random.normal(0, 1, N_STOCKS)
    fake_signal = np.random.normal(0, 1, N_STOCKS)  # 纯随机的伪信号

    # 噪声
    eps = np.random.normal(0, 5, N_STOCKS)

    # 收益率 = 因子暴露 × 因子收益 + 异象效应 + 噪声
    # 真异象：收益率包含 gamma_true * anomaly_signal
    ret_true = (beta_mkt_i * MKT_f[t] + beta_smb_i * SMB_f[t]
                + beta_hml_i * HML_f[t] + gamma_true * anomaly_signal + eps)

    # 伪异象：收益率不包含 fake_signal 的效应 (gamma = 0)
    ret_fake = (beta_mkt_i * MKT_f[t] + beta_smb_i * SMB_f[t]
                + beta_hml_i * HML_f[t] + gamma_fake * fake_signal + eps)

    # --- 真异象：按 anomaly_signal 排序分组 ---
    df_t = pd.DataFrame({'signal': anomaly_signal, 'ret': ret_true})
    df_t['group'] = pd.qcut(df_t['signal'], N_GROUPS,
                            labels=[f'Q{i}' for i in range(1, N_GROUPS+1)])
    group_ret = df_t.groupby('group')['ret'].mean()
    spread_true = group_ret['Q5'] - group_ret['Q1']  # 高信号 - 低信号
    true_anomaly_spreads.append(spread_true)

    # --- 伪异象：按 fake_signal 排序分组 ---
    df_f = pd.DataFrame({'signal': fake_signal, 'ret': ret_fake})
    df_f['group'] = pd.qcut(df_f['signal'], N_GROUPS,
                            labels=[f'Q{i}' for i in range(1, N_GROUPS+1)])
    group_ret_f = df_f.groupby('group')['ret'].mean()
    spread_fake = group_ret_f['Q5'] - group_ret_f['Q1']
    fake_anomaly_spreads.append(spread_fake)

    if t < 3:
        print(f"\n  月份 {t+1}:")
        print(f"    真异象 Spread (Q5-Q1) = {spread_true:.2f}%")
        print(f"    伪异象 Spread (Q5-Q1) = {spread_fake:.2f}%")

true_anomaly_spreads = np.array(true_anomaly_spreads)
fake_anomaly_spreads = np.array(fake_anomaly_spreads)

print(f"\n  ... (省略第 4~{N_MONTHS} 月) ...")
print(f"\n✅ 完成！")
print(f"  真异象 Spread 均值: {true_anomaly_spreads.mean():.3f}%")
print(f"  伪异象 Spread 均值: {fake_anomaly_spreads.mean():.3f}%")

In [ ]:
# ========== 异象检验: 对 FF3 做回归 ==========
print("═" * 60)
print("📋 异象检验: 回归 Spread 对 FF3 因子")
print("═" * 60)

# 构建 FF3 自变量矩阵
X_ff3 = sm.add_constant(np.column_stack([MKT_f, SMB_f, HML_f]))
factor_names = ['Alpha', 'MKT', 'SMB', 'HML']

# ---- 真异象回归 ----
print("\n" + "─" * 60)
print("📊 检验 1: 真异象 (γ = 0.5)")
print("─" * 60)

model_true = sm.OLS(true_anomaly_spreads, X_ff3).fit()

print(f"\n  回归方程: R_spread,t = α + β_MKT·MKT + β_SMB·SMB + β_HML·HML + ε")
print(f"\n  {'参数':>10s}  {'估计值':>10s}  {'标准误':>10s}  {'t值':>10s}  {'P值':>10s}  {'显著?':>6s}")
print(f"  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*6}")
for name, coef, se, tv, pv in zip(factor_names, model_true.params, model_true.bse,
                                   model_true.tvalues, model_true.pvalues):
    sig = '✅' if abs(tv) > 2 else ''
    print(f"  {name:>10s}  {coef:>10.4f}  {se:>10.4f}  {tv:>10.4f}  {pv:>10.6f}  {sig:>6s}")

print(f"\n  R² = {model_true.rsquared:.4f}")
alpha_true = model_true.params[0]
t_alpha_true = model_true.tvalues[0]
p_alpha_true = model_true.pvalues[0]

print(f"\n  🎯 Alpha 检验结果:")
print(f"     α̂ = {alpha_true:.4f}% (月均超额收益)")
print(f"     t_α = {t_alpha_true:.4f}")
print(f"     P 值 = {p_alpha_true:.6f}")
if abs(t_alpha_true) > 2:
    print(f"     ✅ |t| = {abs(t_alpha_true):.2f} > 2 → Alpha 显著！异象成立！")
else:
    print(f"     ❌ |t| = {abs(t_alpha_true):.2f} ≤ 2 → Alpha 不显著")

# ---- 伪异象回归 ----
print("\n" + "─" * 60)
print("📊 检验 2: 伪异象 (γ = 0)")
print("─" * 60)

model_fake = sm.OLS(fake_anomaly_spreads, X_ff3).fit()

print(f"\n  回归方程: R_spread,t = α + β_MKT·MKT + β_SMB·SMB + β_HML·HML + ε")
print(f"\n  {'参数':>10s}  {'估计值':>10s}  {'标准误':>10s}  {'t值':>10s}  {'P值':>10s}  {'显著?':>6s}")
print(f"  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*6}")
for name, coef, se, tv, pv in zip(factor_names, model_fake.params, model_fake.bse,
                                   model_fake.tvalues, model_fake.pvalues):
    sig = '✅' if abs(tv) > 2 else ''
    print(f"  {name:>10s}  {coef:>10.4f}  {se:>10.4f}  {tv:>10.4f}  {pv:>10.6f}  {sig:>6s}")

print(f"\n  R² = {model_fake.rsquared:.4f}")
alpha_fake = model_fake.params[0]
t_alpha_fake = model_fake.tvalues[0]
p_alpha_fake = model_fake.pvalues[0]

print(f"\n  🎯 Alpha 检验结果:")
print(f"     α̂ = {alpha_fake:.4f}% (月均超额收益)")
print(f"     t_α = {t_alpha_fake:.4f}")
print(f"     P 值 = {p_alpha_fake:.6f}")
if abs(t_alpha_fake) > 2:
    print(f"     ⚠️ |t| = {abs(t_alpha_fake):.2f} > 2 → Alpha 显著（意外！可能是随机波动）")
else:
    print(f"     ❌ |t| = {abs(t_alpha_fake):.2f} ≤ 2 → Alpha 不显著 → 不是异象（符合预期）")

In [ ]:
# ========== 手算验证: 真异象的 Alpha ==========
print("🔬 手算验证（真异象 Alpha）:")
print("─" * 55)

# 构造矩阵
X_manual = np.column_stack([np.ones(N_MONTHS), MKT_f, SMB_f, HML_f])
y_manual = true_anomaly_spreads

# OLS: β̂ = (X'X)^{-1} X'y
XtX_m = X_manual.T @ X_manual
XtX_inv_m = np.linalg.inv(XtX_m)
beta_manual = XtX_inv_m @ (X_manual.T @ y_manual)

# 残差和 sigma
resid_m = y_manual - X_manual @ beta_manual
k_m = X_manual.shape[1]
sigma_sq_m = np.sum(resid_m**2) / (N_MONTHS - k_m)
se_alpha_m = np.sqrt(sigma_sq_m * XtX_inv_m[0, 0])
t_alpha_m = beta_manual[0] / se_alpha_m
p_alpha_m = 2 * (1 - stats.t.cdf(abs(t_alpha_m), df=N_MONTHS - k_m))

print(f"  手算 α̂ = {beta_manual[0]:.6f}")
print(f"  statsmodels α̂ = {model_true.params[0]:.6f}")
print(f"  手算 t_α = {t_alpha_m:.6f}")
print(f"  statsmodels t_α = {model_true.tvalues[0]:.6f}")
print(f"  手算 P 值 = {p_alpha_m:.6f}")
print(f"  statsmodels P 值 = {model_true.pvalues[0]:.6f}")
print(f"\n  ✅ 验证通过！")

In [ ]:
# ========== 可视化: 真异象 vs 伪异象 对比 ==========
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
months = np.arange(1, N_MONTHS + 1)

# --- 图1: 真异象 Spread 时间序列 ---
ax1 = axes[0, 0]
ax1.bar(months, true_anomaly_spreads,
        color=['#2ecc71' if s > 0 else '#e74c3c' for s in true_anomaly_spreads],
        alpha=0.7, edgecolor='none')
ax1.axhline(y=true_anomaly_spreads.mean(), color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {true_anomaly_spreads.mean():.2f}%')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('Spread (%)', fontsize=11)
ax1.set_title('TRUE Anomaly: Monthly Spread', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 图2: 伪异象 Spread 时间序列 ---
ax2 = axes[0, 1]
ax2.bar(months, fake_anomaly_spreads,
        color=['#2ecc71' if s > 0 else '#e74c3c' for s in fake_anomaly_spreads],
        alpha=0.7, edgecolor='none')
ax2.axhline(y=fake_anomaly_spreads.mean(), color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {fake_anomaly_spreads.mean():.2f}%')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('Spread (%)', fontsize=11)
ax2.set_title('FAKE Anomaly: Monthly Spread', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

# --- 图3: 真异象回归系数 t 值 ---
ax3 = axes[1, 0]
coef_labels = ['\u03b1', '\u03b2_MKT', '\u03b2_SMB', '\u03b2_HML']
t_vals_true = model_true.tvalues
colors_true = ['#e74c3c' if abs(t) > 2 else '#95a5a6' for t in t_vals_true]
bars3 = ax3.barh(coef_labels, t_vals_true, color=colors_true, edgecolor='black', alpha=0.85)
ax3.axvline(x=2, color='red', linestyle='--', alpha=0.5)
ax3.axvline(x=-2, color='red', linestyle='--', alpha=0.5)
ax3.axvline(x=0, color='black', linewidth=0.8)
for bar, tv in zip(bars3, t_vals_true):
    offset = 0.15 * np.sign(tv)
    ax3.text(tv + offset, bar.get_y() + bar.get_height()/2,
             f'{tv:.2f}', ha='left' if tv >= 0 else 'right', va='center', fontweight='bold')
ax3.set_xlabel('T-statistic', fontsize=11)
ax3.set_title('TRUE Anomaly: Regression T-values', fontsize=13, fontweight='bold')
ax3.grid(axis='x', alpha=0.3)

# --- 图4: 伪异象回归系数 t 值 ---
ax4 = axes[1, 1]
t_vals_fake = model_fake.tvalues
colors_fake = ['#e74c3c' if abs(t) > 2 else '#95a5a6' for t in t_vals_fake]
bars4 = ax4.barh(coef_labels, t_vals_fake, color=colors_fake, edgecolor='black', alpha=0.85)
ax4.axvline(x=2, color='red', linestyle='--', alpha=0.5)
ax4.axvline(x=-2, color='red', linestyle='--', alpha=0.5)
ax4.axvline(x=0, color='black', linewidth=0.8)
for bar, tv in zip(bars4, t_vals_fake):
    offset = 0.15 * np.sign(tv) if tv != 0 else 0.15
    ax4.text(tv + offset, bar.get_y() + bar.get_height()/2,
             f'{tv:.2f}', ha='left' if tv >= 0 else 'right', va='center', fontweight='bold')
ax4.set_xlabel('T-statistic', fontsize=11)
ax4.set_title('FAKE Anomaly: Regression T-values', fontsize=13, fontweight='bold')
ax4.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  上排：Spread 时间序列。真异象均值明显偏离零；伪异象围绕零波动")
print(f"  下排：回归 t 值。红色 = |t| > 2 (显著)")
print(f"    真异象: α 的 t = {t_vals_true[0]:.2f} → Alpha 显著")
print(f"    伪异象: α 的 t = {t_vals_fake[0]:.2f} → Alpha 不显著")

In [ ]:
# ========== 综合对比报告 ==========
print("=" * 60)
print("📋 异象检验综合报告")
print("=" * 60)

print(f"\n🎯 研究问题:")
print(f"   给定 FF3 因子模型，两个候选异象变量是否产生显著 Alpha？")

print(f"\n📊 数据概况:")
print(f"   股票数量: {N_STOCKS} 只/月")
print(f"   时间跨度: {N_MONTHS} 个月")
print(f"   分组方式: {N_GROUPS} 分位，Q5-Q1 Spread")

print(f"\n📊 Spread 统计:")
print(f"   {'指标':>12s}  {'真异象':>10s}  {'伪异象':>10s}")
print(f"   {'─'*12}  {'─'*10}  {'─'*10}")
print(f"   {'Spread均值':>12s}  {true_anomaly_spreads.mean():>10.3f}%  {fake_anomaly_spreads.mean():>10.3f}%")
print(f"   {'Spread标准差':>12s}  {true_anomaly_spreads.std():>10.3f}%  {fake_anomaly_spreads.std():>10.3f}%")

# T 检验 on raw spread (without controlling for factors)
t_raw_true, p_raw_true = stats.ttest_1samp(true_anomaly_spreads, 0)
t_raw_fake, p_raw_fake = stats.ttest_1samp(fake_anomaly_spreads, 0)

print(f"\n🧮 原始 Spread T 检验 (不控制因子):")
print(f"   {'指标':>12s}  {'真异象':>10s}  {'伪异象':>10s}")
print(f"   {'─'*12}  {'─'*10}  {'─'*10}")
print(f"   {'t 统计量':>12s}  {t_raw_true:>10.3f}  {t_raw_fake:>10.3f}")
print(f"   {'P 值':>12s}  {p_raw_true:>10.6f}  {p_raw_fake:>10.6f}")

print(f"\n🧮 Alpha 检验 (控制 FF3 因子):")
print(f"   {'指标':>12s}  {'真异象':>10s}  {'伪异象':>10s}")
print(f"   {'─'*12}  {'─'*10}  {'─'*10}")
print(f"   {'α̂':>12s}  {model_true.params[0]:>10.4f}%  {model_fake.params[0]:>10.4f}%")
print(f"   {'SE(α)':>12s}  {model_true.bse[0]:>10.4f}  {model_fake.bse[0]:>10.4f}")
print(f"   {'t_α':>12s}  {model_true.tvalues[0]:>10.4f}  {model_fake.tvalues[0]:>10.4f}")
print(f"   {'P(α)':>12s}  {model_true.pvalues[0]:>10.6f}  {model_fake.pvalues[0]:>10.6f}")
print(f"   {'R²':>12s}  {model_true.rsquared:>10.4f}  {model_fake.rsquared:>10.4f}")

print(f"\n🎯 结论:")
if abs(model_true.tvalues[0]) > 2:
    print(f"  ✅ 真异象: α = {model_true.params[0]:.3f}%, t = {model_true.tvalues[0]:.2f}")
    print(f"     → Alpha 显著，控制 FF3 后仍有超额收益，异象成立")
else:
    print(f"  ❌ 真异象: α = {model_true.params[0]:.3f}%, t = {model_true.tvalues[0]:.2f}")
    print(f"     → Alpha 不显著")

if abs(model_fake.tvalues[0]) > 2:
    print(f"  ⚠️ 伪异象: α = {model_fake.params[0]:.3f}%, t = {model_fake.tvalues[0]:.2f}")
    print(f"     → Alpha 意外显著（可能是 Type I 错误）")
else:
    print(f"  ❌ 伪异象: α = {model_fake.params[0]:.3f}%, t = {model_fake.tvalues[0]:.2f}")
    print(f"     → Alpha 不显著，不构成异象（符合预期）")

print("\n" + "=" * 60)

In [ ]:
# ========== 可视化: 并排 Alpha 对比 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: Alpha 对比 ---
ax1 = axes[0]
labels_compare = ['TRUE Anomaly', 'FAKE Anomaly']
alphas = [model_true.params[0], model_fake.params[0]]
alpha_ses = [model_true.bse[0], model_fake.bse[0]]
t_crit = stats.t.ppf(0.975, df=N_MONTHS - 4)
ci_lows = [a - t_crit * se for a, se in zip(alphas, alpha_ses)]
ci_highs = [a + t_crit * se for a, se in zip(alphas, alpha_ses)]

colors_alpha = ['#2ecc71' if abs(a/se) > 2 else '#95a5a6' for a, se in zip(alphas, alpha_ses)]
bars1 = ax1.bar(labels_compare, alphas, color=colors_alpha, edgecolor='black', alpha=0.85,
                width=0.5)

# Error bars (confidence intervals)
for i, (label, a, ci_l, ci_h) in enumerate(zip(labels_compare, alphas, ci_lows, ci_highs)):
    ax1.plot([i, i], [ci_l, ci_h], 'k-', linewidth=2)
    ax1.plot([i-0.1, i+0.1], [ci_l, ci_l], 'k-', linewidth=2)
    ax1.plot([i-0.1, i+0.1], [ci_h, ci_h], 'k-', linewidth=2)

for bar, a, tv in zip(bars1, alphas, [model_true.tvalues[0], model_fake.tvalues[0]]):
    va = 'bottom' if a >= 0 else 'top'
    offset = 0.05 if a >= 0 else -0.05
    ax1.text(bar.get_x() + bar.get_width()/2., a + offset,
             f'\u03b1={a:.3f}%\nt={tv:.2f}', ha='center', va=va, fontweight='bold', fontsize=10)

ax1.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='H\u2080: \u03b1=0')
ax1.set_ylabel('Alpha (%)', fontsize=12)
ax1.set_title('Alpha Comparison: True vs Fake Anomaly', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 右图: Spread 分布对比 ---
ax2 = axes[1]
ax2.hist(true_anomaly_spreads, bins=15, alpha=0.6, color='#2ecc71', edgecolor='black',
         label=f'TRUE (mean={true_anomaly_spreads.mean():.2f}%)', density=True)
ax2.hist(fake_anomaly_spreads, bins=15, alpha=0.6, color='#95a5a6', edgecolor='black',
         label=f'FAKE (mean={fake_anomaly_spreads.mean():.2f}%)', density=True)
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero')
ax2.axvline(x=true_anomaly_spreads.mean(), color='#2ecc71', linestyle='--', linewidth=2)
ax2.axvline(x=fake_anomaly_spreads.mean(), color='#95a5a6', linestyle='--', linewidth=2)
ax2.set_xlabel('Spread (%)', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Spread Distribution: True vs Fake', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：Alpha 估计值与 95% 置信区间")
print(f"    绿色 = Alpha 显著 (CI 不含 0)")
print(f"    灰色 = Alpha 不显著 (CI 含 0)")
print(f"  右图：Spread 分布")
print(f"    真异象分布明显右移（远离 0）")
print(f"    伪异象分布以 0 为中心")

---

## 5. 深入理解：Alpha 检验 vs 原始 Spread T 检验

### 💡 关键区别

很多初学者会困惑："直接对 Spread 做 T 检验不就行了？为什么还要回归？"

| 方法 | 公式 | 含义 |
|------|------|------|
| 原始 T 检验 | $t = \bar{R}_{spread} / SE(R_{spread})$ | Spread 是否显著不为零 |
| Alpha 检验 | $t_\alpha = \hat{\alpha} / SE(\hat{\alpha})$ | **控制因子后**，Spread 是否仍然显著 |

```
原始 Spread = α + β₁·MKT + β₂·SMB + β₃·HML + ε
              ↑   ↑─────────────────────────────↑
            Alpha        来自因子暴露的部分

原始 T 检验: 检验整个 Spread 是否 ≠ 0
Alpha 检验:  检验剔除因子暴露后的剩余部分 (α) 是否 ≠ 0
```

**核心差异**：Alpha 检验**更精确**，因为它排除了"借助已知因子暴露赚钱"的可能性。

In [ ]:
# ========== 对比: 原始 T 检验 vs Alpha 检验 ==========
print("📊 原始 T 检验 vs Alpha 检验 对比")
print("═" * 60)

# 原始 T 检验
t_raw_true, p_raw_true = stats.ttest_1samp(true_anomaly_spreads, 0)
t_raw_fake, p_raw_fake = stats.ttest_1samp(fake_anomaly_spreads, 0)

print(f"\n--- 真异象 ---")
print(f"  原始 Spread T 检验:")
print(f"    R̄_spread = {true_anomaly_spreads.mean():.4f}%, t = {t_raw_true:.4f}, P = {p_raw_true:.6f}")
print(f"  Alpha 检验 (控制 FF3):")
print(f"    α̂ = {model_true.params[0]:.4f}%, t = {model_true.tvalues[0]:.4f}, P = {model_true.pvalues[0]:.6f}")
print(f"  💡 两种检验都显著 → 收益既来自因子暴露，也来自独立的 Alpha")

print(f"\n--- 伪异象 ---")
print(f"  原始 Spread T 检验:")
print(f"    R̄_spread = {fake_anomaly_spreads.mean():.4f}%, t = {t_raw_fake:.4f}, P = {p_raw_fake:.6f}")
print(f"  Alpha 检验 (控制 FF3):")
print(f"    α̂ = {model_fake.params[0]:.4f}%, t = {model_fake.tvalues[0]:.4f}, P = {model_fake.pvalues[0]:.6f}")
print(f"  💡 即使原始 Spread 可能不为零，Alpha 检验更精确地告诉我们是否有独立超额收益")

print(f"\n📐 经济显著性 (Sharpe Ratio):")
sr_true = true_anomaly_spreads.mean() / true_anomaly_spreads.std(ddof=1) * np.sqrt(12)
sr_fake = fake_anomaly_spreads.mean() / fake_anomaly_spreads.std(ddof=1) * np.sqrt(12)

# Information Ratio (Alpha-based)
resid_true = model_true.resid
ir_true = model_true.params[0] / resid_true.std(ddof=1) * np.sqrt(12)
resid_fake = model_fake.resid
ir_fake = model_fake.params[0] / resid_fake.std(ddof=1) * np.sqrt(12)

print(f"  {'指标':>20s}  {'真异象':>10s}  {'伪异象':>10s}")
print(f"  {'─'*20}  {'─'*10}  {'─'*10}")
print(f"  {'原始年化SR':>20s}  {sr_true:>10.3f}  {sr_fake:>10.3f}")
print(f"  {'信息比率(IR)':>20s}  {ir_true:>10.3f}  {ir_fake:>10.3f}")
print(f"\n  💡 信息比率 (IR) = Alpha / 残差波动率")
print(f"     IR 更好地衡量了因子模型无法解释的超额收益的质量")

---

## 6. 截面回归检验异象（Fama-MacBeth 方法）

### 📐 为什么需要 FM 回归检验异象？

上面的排序分组（Portfolio Sorts）检验的是**原始 Spread**：多空组合的收益差。但有人质疑："你只是买了小市值、高 B/M 的股票，跟你的异象变量没关系。"

**Portfolio Sorts 的局限**：只能检验单一变量的 Spread，无法同时控制其他变量。

**Fama-MacBeth (FM) 回归的优势**：
- 在截面回归中**同时放入**异象变量 + 控制变量（市值、B/M 等）
- 异象变量的回归系数 $\lambda_1$ 表示**控制其他变量后**，该变量对收益的边际贡献
- 如果 $\lambda_1$ 显著 → 该变量构成**真正的异象**（不是市值或价值效应的伪装）

### 📐 核心步骤

**第 1 步**：对每个月 $t$，做截面回归（Cross-Sectional Regression）：

$$r_{i,t} = \lambda_{0,t} + \lambda_{1,t} \times \text{AnomalyVar}_i + \lambda_{2,t} \times \text{Size}_i + \lambda_{3,t} \times \text{BM}_i + e_{i,t}$$

- $r_{i,t}$ = 股票 $i$ 在 $t$ 月的收益率
- $\text{AnomalyVar}_i$ = 股票 $i$ 的异象变量值（如动量）
- $\text{Size}_i, \text{BM}_i$ = 控制变量（市值、账面市值比）
- $\lambda_{1,t}$ = $t$ 月异象变量的"溢价"（控制 Size 和 BM 后）

**第 2 步**：收集所有月份的 $\lambda_{1,t}$，得到时间序列 $\{\lambda_{1,1}, \lambda_{1,2}, ..., \lambda_{1,T}\}$

**第 3 步**：检验 $\lambda_1$ 的均值是否显著不为零：

$$t = \frac{\bar{\lambda}_1}{\text{SE}_{\text{NW}}(\bar{\lambda}_1)}$$

其中 $\text{SE}_{\text{NW}}$ 是 Newey-West 调整的标准误（处理自相关）。

### 🎯 微型例子：10 只股票 × 3 个月

下面我们用一个极小的例子**手算**整个 FM 回归流程：
- 10 只股票，3 个月
- 每只股票有已知的市值（Size）、账面市值比（BM）和动量信号
- 真实异象效应 $\gamma = 0.05$（每单位动量 → 多赚 0.05%）
- **先手算**，再用 statsmodels 验证

In [ ]:
# ========== 微型 FM 回归示例：10 只股票 × 3 个月 ==========
np.random.seed(42)

# 10 只股票的特征（不随时间变化）
stocks = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']
anomaly_var = np.array([-15, -10, -5, -2, 0, 3, 5, 10, 15, 25], dtype=float)  # 动量信号
size = np.array([100, 200, 150, 300, 50, 250, 180, 120, 350, 80], dtype=float)   # 市值（亿）
bm = np.array([0.8, 0.5, 1.2, 0.3, 1.0, 0.4, 0.7, 1.5, 0.2, 0.9], dtype=float)  # B/M

# 真实参数
gamma_true = 0.05   # 动量效应（每单位动量 → 多赚 0.05%）
beta_size = -0.002  # 市值效应（小市值多赚）
beta_bm = 0.8       # 价值效应（高 B/M 多赚）

# 每个月的因子收益（截距项，模拟市场整体涨跌）
lambda0 = np.array([1.0, 2.0, 0.5])  # 3 个月的截距

# 生成 3 个月的收益数据
returns_3m = np.zeros((10, 3))
noise_3m = np.random.normal(0, 1.5, (10, 3))
for t in range(3):
    returns_3m[:, t] = (lambda0[t]
                        + gamma_true * anomaly_var
                        + beta_size * size
                        + beta_bm * bm
                        + noise_3m[:, t])

# 构建面板数据
panel_rows = []
for t in range(3):
    for i in range(10):
        panel_rows.append({
            'Stock': stocks[i],
            'Month': t + 1,
            'AnomalyVar': anomaly_var[i],
            'Size': size[i],
            'BM': bm[i],
            'Return': returns_3m[i, t]
        })
df_panel = pd.DataFrame(panel_rows)

print("📋 面板数据（前 10 行）：")
print(df_panel.head(10).to_string(index=False))
print(f"\n  ... 共 {len(df_panel)} 行 (10 只股票 × 3 个月)")
print(f"\n📐 真实参数: γ = {gamma_true}, β_size = {beta_size}, β_bm = {beta_bm}")
print(f"   月度截距 λ₀: {lambda0}")

In [ ]:
# ========== 手算 FM 截面回归：逐月 ==========
print("═" * 65)
print("📊 FM 回归第 1 步：逐月截面回归")
print("═" * 65)

lambda1_series = []  # 存储每月的 λ₁（异象变量系数）
lambda0_series = []  # 存储每月的 λ₀（截距）

for t in range(3):
    month_data = df_panel[df_panel['Month'] == t + 1]
    y_t = month_data['Return'].values
    
    # X 矩阵: [1, AnomalyVar, Size, BM]
    X_t = np.column_stack([
        np.ones(10),
        month_data['AnomalyVar'].values,
        month_data['Size'].values,
        month_data['BM'].values
    ])
    
    # 手算 OLS: β̂ = (X'X)^{-1} X'y
    XtX = X_t.T @ X_t
    XtX_inv = np.linalg.inv(XtX)
    beta_hat = XtX_inv @ (X_t.T @ y_t)
    
    lambda0_series.append(beta_hat[0])
    lambda1_series.append(beta_hat[1])
    
    # 用 statsmodels 验证
    X_sm = sm.add_constant(month_data[['AnomalyVar', 'Size', 'BM']].values)
    model_sm = sm.OLS(y_t, X_sm).fit()
    
    print(f"\n  月份 {t+1}:")
    print(f"    截面回归: r_i = λ₀ + λ₁·AnomalyVar + λ₂·Size + λ₃·BM + e")
    print(f"    {'参数':>6s}  {'手算':>10s}  {'statsmodels':>12s}")
    print(f"    {'─'*6}  {'─'*10}  {'─'*12}")
    param_labels = ['λ₀', 'λ₁', 'λ₂', 'λ₃']
    for j, label in enumerate(param_labels):
        print(f"    {label:>6s}  {beta_hat[j]:>10.4f}  {model_sm.params[j]:>12.4f}")
    print(f"    💡 λ₁ = {beta_hat[1]:.4f} → 本月异象变量每增加 1 单位，收益增加 {beta_hat[1]:.4f}%")

lambda1_series = np.array(lambda1_series)
lambda0_series = np.array(lambda0_series)

print(f"\n" + "═" * 65)
print(f"📊 FM 回归第 2-3 步：收集 λ₁ 时间序列，检验均值")
print(f"═" * 65)
print(f"\n  λ₁ 时间序列: {lambda1_series}")
print(f"  λ̄₁ = mean(λ₁) = {lambda1_series.mean():.4f}")
print(f"  真实 γ = {gamma_true}")

# 简单 t 检验（小样本）
se_lambda1 = lambda1_series.std(ddof=1) / np.sqrt(len(lambda1_series))
t_lambda1 = lambda1_series.mean() / se_lambda1
print(f"\n  SE(λ̄₁) = std(λ₁)/√T = {lambda1_series.std(ddof=1):.4f}/√3 = {se_lambda1:.4f}")
print(f"  t(λ̄₁) = {lambda1_series.mean():.4f} / {se_lambda1:.4f} = {t_lambda1:.4f}")
print(f"\n  💡 注意：3 个月数据太少，t 检验不可靠")
print(f"     下面用 60 个月做正式检验")

In [ ]:
# ========== 完整 FM 回归：200 只股票 × 60 个月 ==========
np.random.seed(42)
N_FM = 200   # 每月 200 只股票
T_FM = 60    # 60 个月

# 股票特征（截面固定）
size_fm = np.random.lognormal(mean=4.5, sigma=0.8, size=N_FM)  # 市值
bm_fm = np.random.uniform(0.2, 2.0, N_FM)                       # B/M
anomaly_fm = np.random.normal(0, 1, N_FM)                       # 异象变量（标准化）

# 真实参数
gamma_fm = 0.3    # 异象效应
beta_size_fm = -0.005  # 市值效应
beta_bm_fm = 0.5       # 价值效应

# FM 回归：逐月收集 λ₁
lambda1_fm = []
lambda1_fm_nosize = []  # 不控制 Size 的版本（对比用）

for t in range(T_FM):
    eps_t = np.random.normal(0, 3, N_FM)
    lambda0_t = np.random.normal(1.0, 0.5)  # 本月截距
    ret_t = (lambda0_t
             + gamma_fm * anomaly_fm
             + beta_size_fm * size_fm
             + beta_bm_fm * bm_fm
             + eps_t)
    
    # 完整 FM 回归（控制 Size 和 BM）
    X_full = sm.add_constant(np.column_stack([anomaly_fm, size_fm, bm_fm]))
    model_full = sm.OLS(ret_t, X_full).fit()
    lambda1_fm.append(model_full.params[1])
    
    # 不控制 Size 的版本
    X_nosize = sm.add_constant(np.column_stack([anomaly_fm, bm_fm]))
    model_nosize = sm.OLS(ret_t, X_nosize).fit()
    lambda1_fm_nosize.append(model_nosize.params[1])

lambda1_fm = np.array(lambda1_fm)
lambda1_fm_nosize = np.array(lambda1_fm_nosize)

print("📊 FM 回归结果（60 个月 λ₁ 时间序列）")
print("═" * 60)
print(f"\n  真实异象效应 γ = {gamma_fm}")
print(f"  每月截面回归: r_i = λ₀ + λ₁·AnomalyVar + λ₂·Size + λ₃·BM + e")
print(f"  收集 60 个月的 λ₁ → 检验 mean(λ₁) ≠ 0")

# 检验 mean(λ₁)
mean_l1 = lambda1_fm.mean()
se_l1 = lambda1_fm.std(ddof=1) / np.sqrt(T_FM)
t_l1 = mean_l1 / se_l1

print(f"\n  📐 控制 Size + BM 的 FM 回归:")
print(f"     λ̄₁ = {mean_l1:.4f}")
print(f"     SE(λ̄₁) = {se_l1:.4f}")
print(f"     t(λ̄₁) = {t_l1:.4f}")
if abs(t_l1) > 2:
    print(f"     ✅ |t| = {abs(t_l1):.2f} > 2 → 异象变量溢价显著！")
else:
    print(f"     ❌ |t| = {abs(t_l1):.2f} ≤ 2 → 不显著")

# 对比：不控制 Size
mean_l1_ns = lambda1_fm_nosize.mean()
se_l1_ns = lambda1_fm_nosize.std(ddof=1) / np.sqrt(T_FM)
t_l1_ns = mean_l1_ns / se_l1_ns

print(f"\n  📐 不控制 Size 的 FM 回归（只控制 BM）:")
print(f"     λ̄₁ = {mean_l1_ns:.4f}")
print(f"     t(λ̄₁) = {t_l1_ns:.4f}")
print(f"\n  💡 两种设定下 λ₁ 都显著 → 该变量是稳健的异象")

In [ ]:
# ========== 验证：与纯时序 α 检验对比 ==========
print("═" * 65)
print("🔬 验证：FM 截面回归 vs 纯时序 α 检验")
print("═" * 65)

# --- 纯时序 α 检验 ---
# 按异象变量排序 → 多空组合 → 对因子回归 → 检验 α
# （不控制 Size 和 BM）
spreads_ts = []
for t in range(T_FM):
    eps_t = np.random.normal(0, 3, N_FM)
    lambda0_t = np.random.normal(1.0, 0.5)
    ret_t = (lambda0_t
             + gamma_fm * anomaly_fm
             + beta_size_fm * size_fm
             + beta_bm_fm * bm_fm
             + eps_t)
    
    # 按 anomaly_fm 排序分 5 组
    ranks = pd.Series(anomaly_fm).rank(pct=True)
    q5_ret = ret_t[ranks >= 0.8].mean()  # 最高 20%
    q1_ret = ret_t[ranks <= 0.2].mean()  # 最低 20%
    spreads_ts.append(q5_ret - q1_ret)

spreads_ts = np.array(spreads_ts)

# 对常数回归（等价于 t 检验均值是否为零）
X_ts = sm.add_constant(np.ones(T_FM))
model_ts = sm.OLS(spreads_ts, X_ts).fit()

print(f"\n  📐 方法 1: 纯时序 α 检验（排序分组 → 多空 Spread → t 检验）")
print(f"     每月按异象变量排序，Q5-Q1 Spread")
print(f"     Spread 均值 = {spreads_ts.mean():.4f}%")
print(f"     t(α) = {model_ts.tvalues[0]:.4f}")
print(f"     ⚠️ 该 Spread 混合了：异象效应 + Size效应 + BM效应")

print(f"\n  📐 方法 2: FM 截面回归（同时控制 Size + BM）")
print(f"     λ̄₁ = {lambda1_fm.mean():.4f}（纯异象溢价，剔除了 Size 和 BM 的影响）")
print(f"     t(λ₁) = {t_l1:.4f}")

print(f"\n  📐 对比结果:")
print(f"     {'方法':>25s}  {'估计值':>10s}  {'t 值':>10s}  {'含义':>30s}")
print(f"     {'─'*25}  {'─'*10}  {'─'*10}  {'─'*30}")
print(f"     {'纯时序 Spread t 检验':>25s}  {spreads_ts.mean():>10.4f}  {model_ts.tvalues[0]:>10.4f}  {'混合了 Size+BM 效应':>30s}")
print(f"     {'FM 回归 λ₁ 检验':>25s}  {lambda1_fm.mean():>10.4f}  {t_l1:>10.4f}  {'纯异象溢价（控制后）':>30s}")

print(f"\n  💡 关键结论:")
print(f"     纯时序检验的 Spread 偏大，因为它混入了 Size 和 BM 的效应")
print(f"     FM 回归的 λ₁ 更精确：它表示控制 Size 和 BM 后，")
print(f"     异象变量每增加 1 单位，收益的边际变化")

In [ ]:
# ========== 可视化: FM 回归结果 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
months_fm = np.arange(1, T_FM + 1)

# --- 图1: λ₁ 时间序列柱状图 ---
ax1 = axes[0]
ax1.bar(months_fm, lambda1_fm,
        color=['#2ecc71' if l > 0 else '#e74c3c' for l in lambda1_fm],
        alpha=0.7, edgecolor='none')
ax1.axhline(y=lambda1_fm.mean(), color='blue', linestyle='--', linewidth=2,
            label=f'λ̄₁ = {lambda1_fm.mean():.3f}')
ax1.axhline(y=gamma_fm, color='orange', linestyle=':', linewidth=2,
            label=f'真实 γ = {gamma_fm}')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('λ₁', fontsize=11)
ax1.set_title('FM Regression: λ₁ Time Series', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 图2: FM λ₁ vs 纯时序 Spread 对比 ---
ax2 = axes[1]
ax2.bar(months_fm - 0.15, lambda1_fm, width=0.3,
        color='#3498db', alpha=0.7, edgecolor='none', label='FM λ₁ (控制 Size+BM)')
ax2.bar(months_fm + 0.15, spreads_ts, width=0.3,
        color='#e67e22', alpha=0.7, edgecolor='none', label='Raw Spread (Q5-Q1)')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('Value (%)', fontsize=11)
ax2.set_title('FM λ₁ vs Raw Spread', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

# --- 图3: 汇总对比 ---
ax3 = axes[2]
methods = ['Raw Spread\nt-Test', 'FM λ₁\n(控 Size+BM)']
estimates = [spreads_ts.mean(), lambda1_fm.mean()]
t_stats = [model_ts.tvalues[0], t_l1]
colors_bar = ['#e67e22', '#3498db']

bars = ax3.bar(methods, estimates, color=colors_bar, edgecolor='black', alpha=0.85, width=0.5)
for bar, est, ts in zip(bars, estimates, t_stats):
    va = 'bottom' if est >= 0 else 'top'
    offset = 0.02 if est >= 0 else -0.02
    ax3.text(bar.get_x() + bar.get_width()/2., est + offset,
             f'Est={est:.3f}\nt={ts:.2f}', ha='center', va=va, fontweight='bold', fontsize=10)

ax3.axhline(y=gamma_fm, color='green', linestyle='--', linewidth=2, alpha=0.7,
            label=f'真实 γ = {gamma_fm}')
ax3.axhline(y=0, color='black', linewidth=0.8)
ax3.set_ylabel('Estimate', fontsize=11)
ax3.set_title('Method Comparison', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：FM 回归每月的 λ₁（异象溢价），蓝线 = 均值，橙线 = 真实值")
print(f"  图2：FM λ₁（蓝）vs 纯时序 Spread（橙），FM 剔除了 Size+BM 的干扰")
print(f"  图3：两种方法的估计值对比，FM λ₁ 更接近真实 γ")

### 💡 关键直觉

| 对比维度 | Portfolio Sorts | FM 截面回归 |
|----------|----------------|-------------|
| 控制变量 | 只能单变量排序 | **同时控制**多个变量 |
| 估计量 | Spread (Q5-Q1) | $\lambda_1$（异象变量系数） |
| 混杂问题 | Size/BM 效应混入 Spread | $\lambda_1$ 是"**纯**"异象溢价 |
| 统计检验 | 对 Spread 均值 t 检验 | 对 $\bar{\lambda}_1$ 的 Newey-West t 检验 |
| 适用场景 | 初步筛选 | **正式检验**异象 |

```
Portfolio Sorts: "买了高动量的股票，多赚了 X%"
   → 但 X% 里可能包含小市值和价值效应

FM 回归: "控制市值和 B/M 后，高动量的股票多赚了 λ₁%"
   → λ₁% 是纯粹的动量异象溢价

结论: FM 回归比 Portfolio Sorts 更严谨
      λ₁ 是比 Spread 更可靠的异象度量
```

---

## 7. 核心概念回顾

### 📌 异象 (Anomaly)
- **定义**: 某个变量或策略产生的收益无法被当前因子模型解释
- **判断**: 控制已知因子后，Alpha 显著不为零
- **意义**: 暗示当前因子模型不完备，可能需要新因子

### 📌 Alpha (截距项)
- **定义**: 时间序列回归中因子模型无法解释的超额收益
- **公式**: $R_{spread,t} = \alpha + \sum_k \beta_k F_{k,t} + \varepsilon_t$ 中的 $\alpha$
- **判断**: $|t_\alpha| > 2$（经验法则），Harvey et al. (2016) 建议 $|t_\alpha| > 3$

### 📌 时间序列回归
- **定义**: 将异象组合收益对已知因子做 OLS 回归
- **公式**: $R_{spread,t} = \alpha + \beta_{MKT} \cdot MKT_t + \beta_{SMB} \cdot SMB_t + \beta_{HML} \cdot HML_t + \varepsilon_t$
- **含义**: $\beta_k$ 衡量异象对因子的暴露，$\alpha$ 衡量独立超额收益

### 📌 异象检验完整流程
- **步骤 1**: 按新变量排序 → 分组 → 构建多空组合
- **步骤 2**: 收集多空组合收益时间序列
- **步骤 3**: 对已知因子模型做回归
- **步骤 4**: 检验 $\alpha$ 是否显著不为零

### 📌 真异象 vs 伪异象
- **真异象**: $\gamma \neq 0$，新变量有独立的选股能力，$\alpha$ 显著
- **伪异象**: $\gamma = 0$，新变量没有独立选股能力，$\alpha$ 不显著
- **区分方法**: 只有 Alpha 检验能区分，原始 Spread 检验不能

### 🔗 完整流程
```
新变量 → 排序分组 → 多空组合 → 收益时间序列 → 回归 FF3 → 检验 α
   ↓         ↓          ↓            ↓              ↓         ↓
 候选因子  Q1~Q5    Q5-Q1 Spread   T期数据     OLS回归    |t|>2?
```

### 📝 检验指标汇总

| 概念 | 要点 | 判断标准 |
|------|------|----------|
| Alpha ($\alpha$) | 因子模型无法解释的超额收益 | 回归截距项 |
| Alpha T 检验 | $\alpha$ 是否显著不为零 | $\|t_\alpha\| > 2$ |
| 原始 Spread T 检验 | Spread 是否不为零（不控制因子） | $\|t\| > 2$ |
| $R^2$ | 已知因子能解释多少收益变异 | 越高说明因子暴露越大 |
| 信息比率 (IR) | Alpha 除以残差波动率 | 类似 Sharpe Ratio |

---

## 8. 常见误区

### ❌ 误区 1: Spread 显著就是异象
**✓ 正确理解**: Spread 显著只说明多空组合有收益，但这个收益可能完全来自对已知因子的暴露（如偏向小市值）。必须做 Alpha 检验，控制已知因子后看截距是否显著。

### ❌ 误区 2: Alpha 不显著说明变量没用
**✓ 正确理解**: Alpha 不显著说明该变量的选股能力可以被**当前因子模型**解释。但如果换一个因子模型（如从 FF3 换到 CAPM），结论可能不同。异象是相对于特定因子模型而言的。

### ❌ 误区 3: R² 越高越好
**✓ 正确理解**: 在异象检验中，$R^2$ 高意味着异象组合的收益大部分能被已知因子解释。我们关心的不是 $R^2$，而是**截距 $\alpha$**。$R^2$ 高但 $\alpha$ 显著 → 异象存在于因子暴露无法解释的那一小部分收益中。

### ❌ 误区 4: 只要 |t| > 2 就确认异象
**✓ 正确理解**: 由于大量异象被同时检验（数据挖掘），Harvey, Liu & Zhu (2016) 建议将显著性阈值提高到 $|t| > 3$。在 5% 显著性水平下，每检验 20 个随机变量就有 1 个可能"侥幸"通过 $|t| > 2$ 的检验。

### ❌ 误区 5: 异象检验不需要考虑因子模型的选择
**✓ 正确理解**: 异象是相对于因子模型定义的。同一个变量，在 CAPM 下可能是异象（$\alpha$ 显著），但在 FF5 下可能不是（$\alpha$ 被吸收）。因子模型的选择直接影响结论，必须明确说明用的是哪个基准模型。